# Mimicking Finance -- replication on company data

One holdings parquet is all you need; **no WRDS**. `Run All` executes every configuration.
Each one calls `R.free()` afterwards, so five configs on a 20-core box will not OOM.

## Benchmarks already verified on real WRDS data
10.9M rows, 12,321 funds, 2010-2024, InvTypeCode 401.

| | paper | measured here |
|---|---|---|
| precision (real positions) | -- | 0.5755 (gbm) / 0.5291 (lstm) |
| precision (incl. padding, N=75) | **0.71** | **0.7156** |
| naive (incl. padding) | **0.52** | **0.5208** |
| **Table X Q5-Q1 (tradeable, actual)** | **-0.79** (t=-3.05) | **-0.660** (gbm) / **-0.657** (lstm) |
| Table X Q5-Q1 (tradeable, **frozen**) | -- | **-0.024** (t=-0.15) -- effect disappears |
| Table XII Q1-Q5 (contemporaneous) | +1.06 (t=5.74) | +1.213 (t=5.46) |

**LSTM and GBM reach nearly identical economic conclusions** (-0.657 vs -0.660) even though
the LSTM is less accurate (0.529 vs 0.576). The conclusion is driven by the data and the
timing convention, not by the architecture -- and that was measured, not assumed.

## Three architectures

| `model` | what it is | speed |
|---|---|---|
| `gbm` | gradient boosting; sequence flattened into `y_lag1..4` columns | fastest |
| `lstm` | weight-shared per-position sequence: one sample = one position's last 8 quarters `[T, F]` | medium |
| `panel_lstm` | **the paper's architecture**: one sample = one fund-quarter's whole cross-section, `[T, N, F] -> LSTM(N*F -> numcell) -> [N, 3]` | slowest |

`panel_lstm` also prints the **padding share** -- the slots that lift precision from 0.58 to 0.71.

## Two holding conventions (Table X reports both)

| | meaning |
|---|---|
| **actual** | each quarter uses **that quarter's reported holdings** -> includes the contribution of subsequent trading |
| **frozen** | weights locked at t, each stock compounds on its own (buy and hold) -> measures only the t-dated portfolio |

**The gap between them is the rebalancing contribution.** Measured on WRDS data:
actual **-0.660** (t=-2.68), frozen **-0.024** (t=-0.15) -- **the effect vanishes once
holdings are frozen**.

Reading: the paper's "less predictable managers outperform" comes **entirely from what they
do next**, not from the portfolio they hold now. Predictable managers are not worse stock
pickers; they trade worse afterwards.

## Key switch `use_manager_memory`

`fs_hold_rate` ("this manager never touches this position") raises accuracy, but it quietly
swaps **"trade direction is predictable"** for **"this fund barely trades"**. Low-turnover
funds outperform historically, so **Table X flips sign** (measured: -0.660 -> +0.116).

**Use `False` to replicate the paper**; `True` is kept only as a counter-example.

## Three timing conventions

`accuracy(t)` needs `shares[t+1]`, so it is only knowable at **t+1**; 13F holdings are public
45-60 days after quarter end, so it is only actionable from **t+2**.

- `contemporaneous` acc(t) x t->t+1 -- **overlaps, biased**
- `predictive` acc(t) x t+1->t+2 -- no overlap, ignores the filing delay
- `tradeable` acc(t) x t+2->t+3 -- **truly actionable**

t-statistics are **Newey-West** (`lags = h-1`) because CRET_{0,h} windows overlap;
plain OLS values are reported alongside as `t{h}_ols`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from dataclasses import replace
%load_ext autoreload
%autoreload 2
import company_replication as R
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)

## 0. Base configuration

Point `data_path` at your parquet. If column names differ, edit `col_map`
(keys are the names in *your* file).

In [ ]:
BASE = R.Config(
    data_path = "manager_holdings/master_batches_return_filtered/master_all_funds_add_filter_ivy_rank_active_rank.parquet",
    eval_timing = "predictive",   # headline timing: predictive | tradeable | contemporaneous
    inv_type_codes = (401,),
    max_rank   = 25,           # also the N used by panel_lstm
    template_N = 75,           # template width for the padding calculation
    min_years  = 7, min_holdings = 10,
    window_q   = 28, test_q = 8, step = 8,
    drop_missing_position = True,   # chg_pct == -100% means info missing, not a real sell
    # --- neural nets: sequences are assembled lazily, so full-sample training by default ---
    seq_len = 8, hidden = 64, dropout = 0.25, lr = 3e-3,
    max_epochs = 25, patience = 5, batch = 8192, device = "auto",
    lstm_max_train = None,     # set 300_000 on a slow CPU box (trades accuracy for time)
    lstm_max_rows  = None,
)
BASE

## 1. Build the panel (once, shared by every configuration)

**Check the class balance printed here first.** On WRDS data it is
sell 0.386 / hold 0.292 / buy 0.322, with 23.6% of share changes being **exactly zero**.
If your hold share is far lower (say 5%), `future_1q_shares_change_pct` follows a different
convention and every downstream number needs recalibrating.

In [ ]:
panel = R.load_and_prepare(BASE)
print(f"\npanel {len(panel):,} rows | {panel.fund.nunique():,} funds | {panel.qi.max()+1} quarters")
RESULTS = {}

### Check the volume features

**Look at `pos_to_vol`.** If `volume` is a dollar amount while `shares` is a share count,
the ratio has no physical meaning (it may still predict, but it is not interpretable).

Rule of thumb: a median around 1e-6 means the denominator is dollars and the units do not
line up -- switch to `position_value / volume` for a real days-to-liquidate measure.
`vol_rank` and `amihud` are unaffected either way.

In [ ]:
volf = [c for c in ("log_volume", "vol_rank", "pos_to_vol", "d_log_vol", "amihud")
        if c in panel.columns]
if volf:
    display(panel[volf].describe().loc[["count","mean","std","min","25%","50%","75%","max"]].round(4))
    med = panel["pos_to_vol"].median()
    print(f"pos_to_vol median = {med:.6g}")
    print("-> units look right (volume is a share count)" if med > 1e-3 else
          "-> WARNING: very small; volume may be dollars. Consider position_value / volume")
else:
    print("no volume column -- volume features skipped")
print(f"\nfeatures actually used: {len([f for f in BASE.features if f in panel.columns])}")

In [ ]:
CONFIGS = {
    "A_gbm_no_mem":   dict(model="gbm",        use_manager_memory=False),
    "B_gbm_mem":      dict(model="gbm",        use_manager_memory=True),
    "C_lstm_no_mem":  dict(model="lstm",       use_manager_memory=False),
    "D_lstm_mem":     dict(model="lstm",       use_manager_memory=True),
    "E_panel_no_mem": dict(model="panel_lstm", use_manager_memory=False),
}
CONFIGS

---
# A -- gbm, no manager memory　(fastest baseline; run this first)

Measured on WRDS: accuracy 0.5755, Table X Q5-Q1 (tradeable, actual) = **-0.660 (t=-3.20 OLS,
-2.68 NW)**.

In [ ]:
RESULTS["A_gbm_no_mem"] = R.run_config(panel, replace(BASE, **CONFIGS["A_gbm_no_mem"]), "A_gbm_no_mem")
R.free(RESULTS)          # drop preds so back-to-back configs do not OOM

In [ ]:
r = RESULTS["A_gbm_no_mem"]
display(r["precision_table"].round(4))
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"== Table X  ACTUAL rebalancing / {tm} ==");  display(r["tableX"][tm].round(3))
    print(f"== Table X  FROZEN buy-and-hold / {tm} ==");  display(r["tableX_frozen"][tm].round(3))
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"== Table XII / {tm} ==");                     display(r["tableXII"][tm].round(3))

---
# B -- gbm, **with** manager memory　(counter-example)

Table X is expected to **flip sign**. Measured on WRDS: -0.660 -> **+0.116 (t=+0.82)**.

In [ ]:
RESULTS["B_gbm_mem"] = R.run_config(panel, replace(BASE, **CONFIGS["B_gbm_mem"]), "B_gbm_mem")
R.free(RESULTS)          # drop preds so back-to-back configs do not OOM

In [ ]:
r = RESULTS["B_gbm_mem"]
display(r["precision_table"].round(4))
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"== Table X  ACTUAL rebalancing / {tm} ==");  display(r["tableX"][tm].round(3))
    print(f"== Table X  FROZEN buy-and-hold / {tm} ==");  display(r["tableX_frozen"][tm].round(3))
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"== Table XII / {tm} ==");                     display(r["tableXII"][tm].round(3))

---
# C -- lstm, no manager memory　**(the paper's architecture)**

The paper uses an LSTM, so this is the architecture-level replication.
Measured on WRDS: accuracy 0.5291, Table X Q5-Q1 (tradeable, actual) = **-0.657 (t=-3.33)**
-- essentially identical to gbm's -0.660.

Slower than gbm on a CPU-only box (~2 min/window on a GPU). Set `lstm_max_train=300_000`
if it drags.

In [ ]:
RESULTS["C_lstm_no_mem"] = R.run_config(panel, replace(BASE, **CONFIGS["C_lstm_no_mem"]), "C_lstm_no_mem")
R.free(RESULTS)          # drop preds so back-to-back configs do not OOM

In [ ]:
r = RESULTS["C_lstm_no_mem"]
display(r["precision_table"].round(4))
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"== Table X  ACTUAL rebalancing / {tm} ==");  display(r["tableX"][tm].round(3))
    print(f"== Table X  FROZEN buy-and-hold / {tm} ==");  display(r["tableX_frozen"][tm].round(3))
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"== Table XII / {tm} ==");                     display(r["tableXII"][tm].round(3))

---
# D -- lstm, with manager memory

In [ ]:
RESULTS["D_lstm_mem"] = R.run_config(panel, replace(BASE, **CONFIGS["D_lstm_mem"]), "D_lstm_mem")
R.free(RESULTS)          # drop preds so back-to-back configs do not OOM

In [ ]:
r = RESULTS["D_lstm_mem"]
display(r["precision_table"].round(4))
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"== Table X  ACTUAL rebalancing / {tm} ==");  display(r["tableX"][tm].round(3))
    print(f"== Table X  FROZEN buy-and-hold / {tm} ==");  display(r["tableX_frozen"][tm].round(3))
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"== Table XII / {tm} ==");                     display(r["tableXII"][tm].round(3))

---
# E -- panel_lstm　**(the paper's original `(T, N, F) -> (N x 3)` cross-section)**

One sample = one fund-quarter's **entire cross-section**: column j is the security ranked
j-th in that fund at t, tracked back through time. When the fund holds fewer than N names,
the surplus columns are **padding**.

This config prints the **padding share** -- direct evidence of how much of the paper's 0.71
comes from empty slots. Slowest of the five; shrink `max_rank` if memory is tight.

In [ ]:
RESULTS["E_panel_no_mem"] = R.run_config(panel, replace(BASE, **CONFIGS["E_panel_no_mem"]), "E_panel_no_mem")
R.free(RESULTS)          # drop preds so back-to-back configs do not OOM

In [ ]:
r = RESULTS["E_panel_no_mem"]
display(r["precision_table"].round(4))
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"== Table X  ACTUAL rebalancing / {tm} ==");  display(r["tableX"][tm].round(3))
    print(f"== Table X  FROZEN buy-and-hold / {tm} ==");  display(r["tableX_frozen"][tm].round(3))
for tm in ("predictive", "tradeable", "contemporaneous"):
    print(f"== Table XII / {tm} ==");                     display(r["tableXII"][tm].round(3))

---
# Summary

`X_sign_matches_paper = yes` means that configuration reproduced the direction of the
paper's Table X under **predictive** timing (`Config.eval_timing`) (less predictable funds outperform).
Expect **yes for every config without manager memory**.

In [ ]:
summary = R.summarize(RESULTS)
summary

In [ ]:
cmp = pd.DataFrame([
    {"metric": "precision (incl. padding)", "paper": 0.71,
     **{k: round(v["precision_table"].iloc[2]["precision"], 4) for k, v in RESULTS.items()}},
    {"metric": "naive (incl. padding)", "paper": 0.52,
     **{k: round(v["precision_table"].iloc[2]["naive"], 4) for k, v in RESULTS.items()}},
    {"metric": "Table X Q5-Q1 (predictive, actual)", "paper": -0.79,
     **{k: round(v["tableX"]["predictive"].iloc[-1].CRET_0_4, 3) for k, v in RESULTS.items()}},
    {"metric": "Table X Q5-Q1 (predictive, frozen)", "paper": np.nan,
     **{k: round(v["tableX_frozen"]["predictive"].iloc[-1].CRET_0_4, 3) for k, v in RESULTS.items()}},
    {"metric": "Table XII Q1-Q5 (contemporaneous)", "paper": 1.06,
     **{k: round(v["tableXII"]["contemporaneous"].iloc[-1].mean_qret, 3) for k, v in RESULTS.items()}},
])
cmp

## Save

In [ ]:
import os
os.makedirs("outputs_company", exist_ok=True)
for tag, r in RESULTS.items():
    r["precision_table"].to_csv(f"outputs_company/precision_{tag}.csv", index=False)
    for tm in ("predictive", "tradeable", "contemporaneous"):
        r["tableX"][tm].to_csv(f"outputs_company/tableX_actual_{tag}_{tm}.csv", index=False)
        r["tableX_frozen"][tm].to_csv(f"outputs_company/tableX_frozen_{tag}_{tm}.csv", index=False)
        r["tableXII"][tm].to_csv(f"outputs_company/tableXII_{tag}_{tm}.csv", index=False)
summary.to_csv("outputs_company/summary.csv", index=False)
cmp.to_csv("outputs_company/compare_with_paper.csv", index=False)
print("saved to outputs_company/")

## If memory runs short

On a 20-core box, if a configuration OOMs:

```python
R.free(RESULTS)                                          # keep tables, drop prediction detail

c = replace(BASE, model="panel_lstm", max_rank=15)       # smaller N -> smaller tensor
c = replace(BASE, model="lstm", lstm_max_train=300_000)  # cap train sequences per window
c = replace(BASE, model="lstm", lstm_max_rows=2_000_000) # subsample the panel first
```

Rough footprints on a 10M-row panel: `gbm` ~3 GB, `lstm` ~2 GB (lazy indices),
`panel_lstm` scales with `max_rank`.

Falling back to `gbm` alone is fine -- the economic conclusions are the same.